# ZINC dense checkpoint trajectory — canonical scores

Runs epochs 10, 100, 250, 500, 1000, and 1990 sequentially for the seed-0 dense model. `run` verifies/setup assets, resumes or skips every epoch after fresh cache validation, computes canonical scores only, releases memory between checkpoints, and writes the dense distribution plot plus CSVs to Drive. Run all cells on a GPU runtime.

In [ ]:
MODE = "run"  # @param ["run", "plot", "status", "setup"]
ARCHITECTURE = "dense"
DRIVE_FOLDER = "/content/drive/MyDrive/graph_specialisation_metrics/multi_seed_models/multiple_checkpoints_zinc"

# Zero selects the GPU-aware profile: A100-80 -> 48, A100-40 -> 24.
GRAPHS_PER_BATCH = 0  # @param {type:"integer"}
ACCELERATOR = "cuda:0"
STRICT_AUDITS = False
RECLAIM_SETUP_LOCK = False
RECLAIM_EPOCH = -1  # @param {type:"integer"}

REPO_URL = "https://github.com/joshgreenwa/Graph-Specialisation-and-Metrics.git"
REPO_REVISION = "expansion/carriage_experiments"
REPO_DIR = "/content/Graph-Specialisation-and-Metrics"
GITHUB_SECRET = "dissertation_key"

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
token = userdata.get(GITHUB_SECRET) or os.environ.get(GITHUB_SECRET)
suffix = REPO_URL.removeprefix("https://github.com/")
clone_url = (
    f"https://x-access-token:{quote(str(token).strip(), safe='')}@github.com/{suffix}"
    if token else REPO_URL
)
repo = Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", clone_url, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "remote", "set-url", "origin", clone_url], check=True)
subprocess.run(["git", "-C", str(repo), "fetch", "origin", REPO_REVISION], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", str(repo), "remote", "set-url", "origin", REPO_URL], check=True)
for entry in (str(repo), str(repo / "src")):
    if entry in sys.path:
        sys.path.remove(entry)
    sys.path.insert(0, entry)
importlib.invalidate_caches()
controller_module = "experiments.methodology.zinc_checkpoint_trajectory_colab"
for name in tuple(sys.modules):
    if name in {controller_module, "graph_specialisation_metrics"} or name.startswith("graph_specialisation_metrics."):
        del sys.modules[name]
methodology_package = sys.modules.get("experiments.methodology")
if methodology_package is not None:
    vars(methodology_package).pop("zinc_checkpoint_trajectory_colab", None)

In [ ]:
from experiments.methodology.zinc_checkpoint_trajectory_colab import run_frontend

result = run_frontend(
    mode=MODE,
    architecture=ARCHITECTURE,
    drive_folder=DRIVE_FOLDER,
    graphs_per_batch=(GRAPHS_PER_BATCH or None),
    accelerator=ACCELERATOR,
    strict_audits=STRICT_AUDITS,
    reclaim_setup_lock=RECLAIM_SETUP_LOCK,
    reclaim_epoch=RECLAIM_EPOCH,
)
result